# Emanuele Imperiali - HW 1 


# 1

Daily volatility:

$$
\sigma_{1d} = 2\% = 0.02
$$

Over 3 days:

$$
\sigma_{3d} = \sigma_{1d}\sqrt{3}
$$


In [ ]:
import numpy as np

sigma_daily = 0.02

sigma_3days = sigma_daily * np.sqrt(3)

print(f"3-day volatility: {sigma_3days:.4%}")

# 2

Black-Scholes call price:

$$
C = S_0N(d_1) - Ke^{-rT}N(d_2)
$$

with

$$
d_1 = \frac{\ln(S_0/K)+(r+\frac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}
$$

$$
d_2 = d_1-\sigma\sqrt{T}
$$

The implied volatility is the value of $\sigma$ such that:

$$
C_{BS}(\sigma) - C_{market} = 0
$$


In [ ]:
from scipy.stats import norm
from scipy.optimize import brentq

S0 = 120
K = 115
T = 0.5
r = 0.03
C_market = 8.75

def call_price(sigma):
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def objective(sigma):
    return call_price(sigma) - C_market

implied_vol = brentq(objective, 0.0001, 5.0)

print(f"Implied volatility: {implied_vol:.4%}")

# 3

EWMA model:

$$
\sigma_t^2 = \lambda \sigma_{t-1}^2 + (1-\lambda)X_t^2
$$

where

$$
X_t = \ln\left(\frac{S_t}{S_{t-1}}\right)
$$


In [ ]:
sigma_old = 0.015
S_old = 30.00
S_new = 30.50
lam = 0.94

X = np.log(S_new / S_old)

variance_new = lam * sigma_old**2 + (1 - lam) * X**2
sigma_new = np.sqrt(variance_new)

print(f"Daily return: {X:.4%}")
print(f"Updated volatility: {sigma_new:.4%}")

# 4

Return:

$$
X_t = \ln\left(\frac{298}{300}\right)
$$

## (a)

EWMA:

$$
\sigma_t^2 = \lambda \sigma_{t-1}^2 + (1-\lambda)X_t^2
$$

## (b)

GARCH(1,1):

$$
\sigma_t^2 = \alpha_0 + \alpha_1X_t^2 + \beta_1\sigma_{t-1}^2
$$


In [ ]:
S_old = 300
S_new = 298
sigma_old = 0.013

X = np.log(S_new / S_old)

# (a) EWMA
lam = 0.94

variance_ewma = lam * sigma_old**2 + (1 - lam) * X**2
sigma_ewma = np.sqrt(variance_ewma)

# (b) GARCH(1,1)
alpha0 = 0.000002
alpha1 = 0.04
beta1 = 0.94

variance_garch = alpha0 + alpha1 * X**2 + beta1 * sigma_old**2
sigma_garch = np.sqrt(variance_garch)

print(f"Daily return: {X:.4%}")
print(f"EWMA volatility: {sigma_ewma:.4%}")
print(f"GARCH volatility: {sigma_garch:.4%}")

# 5

GARCH(1,1):

$$
\sigma_{t+\Delta}^2 = \alpha_0 + \alpha_1X_t^2 + \beta_1\sigma_t^2
$$

with

$$
\alpha_0 = 0.0000013465,\qquad
\alpha_1 = 0.083394,\qquad
\beta_1 = 0.910116
$$

## (a)

$$
\alpha_0 = (1-\alpha_1-\beta_1)V_L
$$

so

$$
V_L = \frac{\alpha_0}{1-\alpha_1-\beta_1}
$$

and

$$
\sigma_L = \sqrt{V_L}
$$


In [ ]:
alpha0 = 0.0000013465
alpha1 = 0.083394
beta1 = 0.910116

VL = alpha0 / (1 - alpha1 - beta1)
sigma_L = np.sqrt(VL)

print(f"Long-term variance: {VL:.8f}")
print(f"Long-term volatility: {sigma_L:.4%}")

## (b)

Since

$$
E[X_t]=0
$$

and

$$
Var(X_t)=\sigma_t^2,
$$

we have

$$
E[X_t^2]=\sigma_t^2.
$$

Therefore,

$$
E[\sigma_{t+n\Delta}^2]
=
V_L+(\alpha_1+\beta_1)^n(\sigma_t^2-V_L)
$$

The volatility forecast is the square root of the expected variance.


In [ ]:
sigma_today = 0.01732
a = alpha1 + beta1

def expected_variance(n):
    return VL + a**n * (sigma_today**2 - VL)

variance_10 = expected_variance(10)
variance_500 = expected_variance(500)

sigma_10 = np.sqrt(variance_10)
sigma_500 = np.sqrt(variance_500)

print(f"Expected volatility after 10 days: {sigma_10:.4%}")
print(f"Expected volatility after 500 days: {sigma_500:.4%}")

## (c)

From

$$
E[\sigma_{t+n\Delta}^2]
=
V_L+(\alpha_1+\beta_1)^n(\sigma_t^2-V_L),
$$

and since

$$
\alpha_1+\beta_1 = 0.99351 < 1,
$$

we have

$$
\lim_{n\to\infty}(\alpha_1+\beta_1)^n = 0.
$$

Therefore,

$$
\lim_{n\to\infty}E[\sigma_{t+n\Delta}^2] = V_L.
$$

Hence the volatility forecast converges to

$$
\sqrt{V_L}=\sigma_L.
$$
